<a href="https://colab.research.google.com/github/suparuek2405/thai-bank-rag-qa/blob/main/notebooks/NB01_document_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# NB01 — Document Processing
# Thai Bank Financial Q&A System

In [1]:
#Install dependencies
!pip install pymupdf langchain langchain-community -q

import fitz
import langchain
print(f"PyMuPDF:   {fitz.__version__}")
print(f"LangChain: {langchain.__version__}")

PyMuPDF:   1.27.2.3
LangChain: 1.3.4


In [14]:
#Mount Google Drive and set paths

from google.colab import drive
drive.mount('/content/drive')

import sys, os

DRIVE_ROOT   = "/content/drive/MyDrive/Github experiment/thai-bank-rag-qa"
PDF_DIR      = f"{DRIVE_ROOT}/data/raw"
PROCESSED_DIR = "data/processed"   # saved locally in Colab runtime


#Clone or pull the GitHub repo first, then run this cell.
REPO_DIR = "/content/thai-bank-rag-qa"

# ── Clone repo (or pull latest if already cloned)
if not os.path.exists(REPO_DIR):
    get_ipython().system(
        "git clone https://github.com/suparuek2405/thai-bank-rag-qa.git /content/thai-bank-rag-qa"
    )
else:
    get_ipython().system("cd /content/thai-bank-rag-qa && git pull --rebase origin main")

sys.path.insert(0, REPO_DIR)

#check: list PDFs on Drive
pdf_files = sorted([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])
print(f"Found {len(pdf_files)} PDFs in Drive:")
for f in pdf_files:
    size_mb = os.path.getsize(os.path.join(PDF_DIR, f)) / 1_000_000
    print(f"  {f}  ({size_mb:.1f} MB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
From https://github.com/suparuek2405/thai-bank-rag-qa
 * branch            main       -> FETCH_HEAD
Already up to date.
Found 10 PDFs in Drive:
  BAY_56-1_2025.pdf  (34.2 MB)
  BBL_56-1_2025.pdf  (20.0 MB)
  CREDIT_56-1_2025.pdf  (18.3 MB)
  KBANK_56-1_2025.pdf  (26.3 MB)
  KKP_56-1_2025.pdf  (11.8 MB)
  KTB_56-1_2025.pdf  (17.0 MB)
  LHFG_56-1_2025.pdf  (29.8 MB)
  SCBX_56-1_2025.pdf  (19.0 MB)
  TISCO_56-1_2025.pdf  (8.4 MB)
  TTB_56-1_2025.pdf  (20.5 MB)


In [ ]:
# clear
import subprocess
result = subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"],
    cwd="/content/thai-bank-rag-qa",
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [15]:
# Goal: see what raw text looks like before chunking.

from src.loader import load_bank_pdf, inspect_page, extraction_stats

# Pick any bank to inspect first
TEST_BANK = "KBANK"

pages = load_bank_pdf(TEST_BANK, PDF_DIR)
print(f"\n{TEST_BANK}: {len(pages)} text-bearing pages")

# Preview page 1 and a middle page
print("\n=== Page 1 ===")
inspect_page(pages[0])

print("\n=== Page 10 ===")
inspect_page(pages[9])


KBANK: 180 text-bearing pages

=== Page 1 ===
Bank: KBANK | Page: 1 | Chars: 1,616
------------------------------------------------------------
Annual Registration Statement
and Annual Report 2025
(Form 56-1 One Report)
Bank of Sustainability
Contents
KASIKORNBANK
Form 56-1 One Report 
2025
Statement of the Board of Directors
Statement of the Chief Executive Officer
Part 1 Business Operations and Performance
1.	Structure and Business Operations of
	 KASIKORNBANK FINANCIAL CONGLOMERATE
2.	Risk Management
3.	Steering Business towards Sustainability
4.	Management Discussion and Analysis
5.	General Information and Other Important Informatio
...

=== Page 10 ===
Bank: KBANK | Page: 10 | Chars: 11,037
------------------------------------------------------------
Subject
Details
Lending
• BOT Circular No. Wor.1965/2568 Re: Submission of the BOT Guidelines on Lending to Certain Business 
Sectors, effective March 12, 2025. The circular introduces additional expectations for financial institutio

In [16]:
#Load ALL 10 banks and print extraction stats

from src.loader import load_all_banks

BANKS = ["BBL", "KBANK", "KTB", "SCBX", "TTB", "TISCO", "BAY", "LHFG", "CREDIT", "KKP"]

print("Loading all 10 bank PDFs...\n")
all_pages = load_all_banks(PDF_DIR, BANKS)

# Summary table
stats = extraction_stats(all_pages)
print(f"\n{'Bank':<8} {'Pages':>6} {'Total chars':>14} {'Avg chars/page':>15}")
print("-" * 47)
for bank, s in stats.items():
    print(f"{bank:<8} {s['total_pages']:>6,} {s['total_chars']:>14,} {s['avg_chars_per_page']:>15,.0f}")

Loading all 10 bank PDFs...

  [BBL] 233 pages | 732,331 chars
  [KBANK] 180 pages | 1,204,229 chars
  [KTB] 393 pages | 1,315,530 chars
  [SCBX] 361 pages | 1,004,045 chars
  [TTB] 195 pages | 499,021 chars
  [TISCO] 364 pages | 898,914 chars
  [BAY] 487 pages | 1,038,311 chars
  [LHFG] 329 pages | 870,815 chars
  [CREDIT] 302 pages | 706,094 chars
  [KKP] 438 pages | 1,343,721 chars

All 10 banks loaded successfully.

Bank      Pages    Total chars  Avg chars/page
-----------------------------------------------
BBL         233        732,331           3,143
KBANK       180      1,204,229           6,690
KTB         393      1,315,530           3,347
SCBX        361      1,004,045           2,781
TTB         195        499,021           2,559
TISCO       364        898,914           2,470
BAY         487      1,038,311           2,132
LHFG        329        870,815           2,647
CREDIT      302        706,094           2,338
KKP         438      1,343,721           3,068


In [18]:
# Find pages that likely contain financial tables (look for "%" or "million")

def find_financial_pages(pages, keywords=["NPL", "NIM", "ROE", "Capital", "Million", "Billion"], n=3):
    """Return up to n pages that contain any of the keywords."""
    hits = []
    for p in pages:
        if any(kw in p["text"] for kw in keywords):
            hits.append(p)
        if len(hits) >= n:
            break
    return hits

for bank in ["KBANK", "KTB", "CREDIT"]:
    hits = find_financial_pages(all_pages[bank])
    if hits:
        print(f"\n=== {bank} — Financial page sample ===")
        inspect_page(hits[0], n_chars=600)


=== KBANK — Financial page sample ===
Bank: KBANK | Page: 4 | Chars: 8,328
------------------------------------------------------------
As of or for the years ended December 31,
2025
2024 
(Restated)
2023
2022
2021
PERFORMANCE INDICATORS
Return on average assets (ROA)
1.11%
1.15%
0.99%
0.86%
0.98%
Return on average equity (ROE) 6
8.62%
9.13%
8.29%
7.38%
8.44%
Net interest margin (NIM)
3.23%
3.60%
3.66%
3.33%
3.21%
Cost to income ratio
43.56%
42.50%
44.10%
43.15%
43.49%
ASSET QUALITY RATIOS / FINANCIAL POLICY RATIOS
Loans to deposits ratio
86.89%
91.36%
92.25%
90.77%
93.20%
NPL gross to total loans 7
3.20%
3.20%
3.19%
3.19%
3.76%
Total allowance for expected credit loss to NPL gross (Coverage ratio)
162.75%
152.34%
152.23%
154.
...

=== KTB — Financial page sample ===
Bank: KTB | Page: 9 | Chars: 2,242
------------------------------------------------------------
2024
2025
2024
2025
3,740,468
Total Assets
(million baht)
3,933,319
46,154 
48,229
21.42%
22.12%
Total Capital
Ratio
Net Prof

In [22]:
# Explore chunking: compare 3 chunk sizes on one page
# Goal: see which size keeps financial sentences intact.

from src.loader import chunk_page

# Use a financial page from KBANK for the comparison
sample_page = find_financial_pages(all_pages["KBANK"])[0]

print(f"Source page: {sample_page['bank_name']} p.{sample_page['page_number']} "
      f"({sample_page['char_count']} chars)\n")

for chunk_size, overlap in [(256, 50), (512, 100), (1024, 100)]:
    chunks = chunk_page(sample_page, chunk_size=chunk_size, overlap=overlap)
    print(f"chunk_size={chunk_size:4d}, overlap={overlap:3d}  →  {len(chunks)} chunks")

    # Show the first chunk so we can compare content boundaries
    print(f"  [First chunk preview] {chunks[0]['text'][:1024]!r}")
    print(f"  [Second chunk preview] {chunks[1]['text'][:1024]!r}")
    print(f"  [Third chunk preview] {chunks[2]['text'][:1024]!r}")
    print()

Source page: KBANK p.4 (8328 chars)

chunk_size= 256, overlap= 50  →  43 chunks
  [First chunk preview] 'As of or for the years ended December 31,\n2025\n2024 \n(Restated)\n2023\n2022\n2021\nPERFORMANCE INDICATORS\nReturn on average assets (ROA)\n1.11%\n1.15%\n0.99%\n0.86%\n0.98%\nReturn on average equity (ROE) 6\n8.62%\n9.13%\n8.29%\n7.38%\n8.44%\nNet interest margin (NIM)\n3.23'
  [Second chunk preview] '%\n8.29%\n7.38%\n8.44%\nNet interest margin (NIM)\n3.23%\n3.60%\n3.66%\n3.33%\n3.21%\nCost to income ratio\n43.56%\n42.50%\n44.10%\n43.15%\n43.49%\nASSET QUALITY RATIOS / FINANCIAL POLICY RATIOS\nLoans to deposits ratio\n86.89%\n91.36%\n92.25%\n90.77%\n93.20%\nNPL gross to total'
  [Third chunk preview] '89%\n91.36%\n92.25%\n90.77%\n93.20%\nNPL gross to total loans 7\n3.20%\n3.20%\n3.19%\n3.19%\n3.76%\nTotal allowance for expected credit loss to NPL gross (Coverage ratio)\n162.75%\n152.34%\n152.23%\n154.26%\n159.08%\nExpected credit loss to average loans (Credit cost) \n1.6'

chunk_s

In [23]:
# Chunk all banks with 3 configs and count totals
# We'll choose the best one based on RAGAS scores — not guessing now.

from src.loader import chunk_documents, save_chunks

CONFIGS = [
    {"chunk_size": 256,  "overlap": 50},
    {"chunk_size": 512,  "overlap": 100},
    {"chunk_size": 1024, "overlap": 100},
]

chunk_sets = {}

for cfg in CONFIGS:
    cs, ov = cfg["chunk_size"], cfg["overlap"]
    print(f"\n── Config: chunk_size={cs}, overlap={ov} ──")
    chunks = chunk_documents(all_pages, chunk_size=cs, overlap=ov)
    chunk_sets[(cs, ov)] = chunks
    save_chunks(chunks, PROCESSED_DIR, chunk_size=cs, overlap=ov)

print("\nAll configs saved to", PROCESSED_DIR)


── Config: chunk_size=256, overlap=50 ──
  [BBL] 233 pages → 3865 chunks (avg 16 chunks/page)
  [KBANK] 180 pages → 6361 chunks (avg 35 chunks/page)
  [KTB] 393 pages → 7167 chunks (avg 18 chunks/page)
  [SCBX] 361 pages → 5386 chunks (avg 14 chunks/page)
  [TTB] 195 pages → 2796 chunks (avg 14 chunks/page)
  [TISCO] 364 pages → 4896 chunks (avg 13 chunks/page)
  [BAY] 487 pages → 5572 chunks (avg 11 chunks/page)
  [LHFG] 329 pages → 4588 chunks (avg 13 chunks/page)
  [CREDIT] 302 pages → 3743 chunks (avg 12 chunks/page)
  [KKP] 438 pages → 7300 chunks (avg 16 chunks/page)

Total: 51674 chunks across 10 banks
Saved 51,674 chunks → data/processed/chunks_c256_o50.json (21.3 MB)

── Config: chunk_size=512, overlap=100 ──
  [BBL] 233 pages → 2068 chunks (avg 8 chunks/page)
  [KBANK] 180 pages → 3399 chunks (avg 18 chunks/page)
  [KTB] 393 pages → 3878 chunks (avg 9 chunks/page)
  [SCBX] 361 pages → 2907 chunks (avg 8 chunks/page)
  [TTB] 195 pages → 1512 chunks (avg 7 chunks/page)
  [TISC

In [24]:
# Verify metadata on a sample chunk
# Metadata is what enables bank-level filtering later.
# Every chunk MUST have: bank_name, source_file, page_number, chunk_index.

sample_chunks = chunk_sets[(512, 100)]

print("Sample chunk (first 3 fields shown as metadata, text truncated):\n")
for chunk in sample_chunks[:3]:
    print({k: v for k, v in chunk.items() if k != "text"})
    print(f"  text preview: {chunk['text'][:100]!r}")
    print()

# Verify all banks present
banks_in_chunks = set(c["bank_name"] for c in sample_chunks)
print(f"Banks in chunk set: {sorted(banks_in_chunks)}")
assert banks_in_chunks == set(BANKS), f"Missing banks: {set(BANKS) - banks_in_chunks}"
print("✓ All 10 banks present in chunk set")

Sample chunk (first 3 fields shown as metadata, text truncated):

{'bank_name': 'BBL', 'source_file': 'BBL_56-1_2025.pdf', 'page_number': 1, 'chunk_index': 0, 'char_count': 148}
  text preview: 'Creating \nOpportunities \nTogether\nAnnual Registration Statement/\nAnnual Report 2025 \n(Form 56-1 One '

{'bank_name': 'BBL', 'source_file': 'BBL_56-1_2025.pdf', 'page_number': 2, 'chunk_index': 0, 'char_count': 253}
  text preview: 'In remembrance of \nHer Majesty Queen Sirikit \nThe Queen Mother, \nwhose enduring compassion \nand devo'

{'bank_name': 'BBL', 'source_file': 'BBL_56-1_2025.pdf', 'page_number': 3, 'chunk_index': 0, 'char_count': 300}
  text preview: 'Vision\nTo be a bank which provides quality financial \nservices in line with customers’ requirements,'

Banks in chunk set: ['BAY', 'BBL', 'CREDIT', 'KBANK', 'KKP', 'KTB', 'LHFG', 'SCBX', 'TISCO', 'TTB']
✓ All 10 banks present in chunk set


In [25]:
# Summary stats per bank (512-char config)

from collections import Counter

chunks_512 = chunk_sets[(512, 100)]
per_bank = Counter(c["bank_name"] for c in chunks_512)

print(f"Chunk distribution (chunk_size=512, overlap=100):\n")
print(f"{'Bank':<8} {'Chunks':>8} {'Share':>8}")
print("-" * 26)
total = sum(per_bank.values())
for bank in BANKS:
    n = per_bank.get(bank, 0)
    print(f"{bank:<8} {n:>8,} {n/total:>7.1%}")
print(f"{'TOTAL':<8} {total:>8,}")

Chunk distribution (chunk_size=512, overlap=100):

Bank       Chunks    Share
--------------------------
BBL         2,068    7.4%
KBANK       3,399   12.2%
KTB         3,878   13.9%
SCBX        2,907   10.5%
TTB         1,512    5.4%
TISCO       2,614    9.4%
BAY         3,006   10.8%
LHFG        2,469    8.9%
CREDIT      2,034    7.3%
KKP         3,926   14.1%
TOTAL      27,813
